In [ ]:
# ============================================================
# WORKSHEET 5 - FULL EXERCISE CODE IN ONE
# End-to-End CNN Model for Image Classification using Keras
# ============================================================

import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report

# -----------------------------
# 1. DEFINE PATHS AND SETTINGS
# -----------------------------
train_dir = "dataset/train"
test_dir = "dataset/test"

img_height = 128
img_width = 128
batch_size = 16
validation_split = 0.2
epochs = 250
seed = 123

# -----------------------------
# 2. TASK 1: VISUALIZE IMAGES
# -----------------------------
class_names = sorted([
    d for d in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, d))
])

print("Classes found:", class_names)

num_classes = len(class_names)
samples_to_show = len(class_names)

plt.figure(figsize=(15, 8))

for i, class_name in enumerate(class_names):
    class_path = os.path.join(train_dir, class_name)
    image_files = [
        f for f in os.listdir(class_path)
        if os.path.isfile(os.path.join(class_path, f))
    ]

    if len(image_files) == 0:
        continue

    random_image = random.choice(image_files)
    image_path = os.path.join(class_path, random_image)

    try:
        img = Image.open(image_path)
        plt.subplot(2, (samples_to_show + 1) // 2, i + 1)
        plt.imshow(img)
        plt.title(class_name)
        plt.axis("off")
    except Exception as e:
        print(f"Could not display image {image_path}: {e}")

plt.tight_layout()
plt.show()

print("\nObservation:")
print("The dataset contains different fruit classes stored in separate folders.")
print("Each class has visually distinct images, though some classes may look similar.")
print("This makes it suitable for image classification using CNN.")

# -----------------------------------
# 3. CHECK AND REMOVE CORRUPTED IMAGES
# -----------------------------------
def remove_corrupted_images(folder):
    corrupted_images = []

    class_dirs = [
        d for d in os.listdir(folder)
        if os.path.isdir(os.path.join(folder, d))
    ]

    for class_name in class_dirs:
        class_path = os.path.join(folder, class_name)

        for filename in os.listdir(class_path):
            image_path = os.path.join(class_path, filename)

            if not os.path.isfile(image_path):
                continue

            try:
                with Image.open(image_path) as img:
                    img.verify()
            except (IOError, SyntaxError, OSError) as e:
                corrupted_images.append(image_path)
                try:
                    os.remove(image_path)
                    print(f"Removed corrupted image: {image_path}")
                except Exception as remove_error:
                    print(f"Could not remove {image_path}: {remove_error}")

    if len(corrupted_images) == 0:
        print("No Corrupted Images Found.")

    return corrupted_images

print("\nChecking train directory for corrupted images...")
remove_corrupted_images(train_dir)

print("\nChecking test directory for corrupted images...")
remove_corrupted_images(test_dir)

# -----------------------------------
# 4. LOAD AND PREPROCESS THE DATA
# -----------------------------------
rescale = tf.keras.layers.Rescaling(1.0 / 255)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="int",
    image_size=(img_height, img_width),
    interpolation="nearest",
    batch_size=batch_size,
    shuffle=True,
    validation_split=validation_split,
    subset="training",
    seed=seed
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="int",
    image_size=(img_height, img_width),
    interpolation="nearest",
    batch_size=batch_size,
    shuffle=False,
    validation_split=validation_split,
    subset="validation",
    seed=seed
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    labels="inferred",
    label_mode="int",
    image_size=(img_height, img_width),
    interpolation="nearest",
    batch_size=batch_size,
    shuffle=False
)

# Normalize pixel values
train_ds = train_ds.map(lambda x, y: (rescale(x), y))
val_ds = val_ds.map(lambda x, y: (rescale(x), y))
test_ds = test_ds.map(lambda x, y: (rescale(x), y))

# Improve performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print("\nDetected class names from dataset:")
print(train_ds.class_names)

num_classes = len(train_ds.class_names)

# -----------------------------------
# 5. TASK 3: BUILD CNN MODEL
# -----------------------------------
model = keras.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),

    # Convolutional Layer 1
    layers.Conv2D(32, (3, 3), strides=1, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2), strides=2),

    # Convolutional Layer 2
    layers.Conv2D(32, (3, 3), strides=1, padding="same", activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2), strides=2),

    # Fully Connected Layers
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

print("\nMODEL SUMMARY:")
model.summary()

# -----------------------------------
# 6. TASK 4: COMPILE MODEL
# -----------------------------------
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# -----------------------------------
# 7. TASK 4: TRAIN MODEL
# -----------------------------------
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="best_cnn_model.keras",
        monitor="val_loss",
        save_best_only=True
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks,
    verbose=1
)

# -----------------------------------
# 8. VISUALIZE TRAINING PROGRESS
# -----------------------------------
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]
train_acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_loss, label="Training Loss")
plt.plot(val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_acc, label="Training Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

# -----------------------------------
# 9. TASK 5: EVALUATE MODEL
# -----------------------------------
test_loss, test_acc = model.evaluate(test_ds, verbose=2)
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# -----------------------------------
# 10. TASK 6: SAVE AND LOAD MODEL
# -----------------------------------
model.save("fruit_cnn_model.h5")
print("\nModel saved as fruit_cnn_model.h5")

loaded_model = tf.keras.models.load_model("fruit_cnn_model.h5")
print("Saved model loaded successfully.")

loaded_loss, loaded_acc = loaded_model.evaluate(test_ds, verbose=0)
print(f"Loaded Model Test Loss: {loaded_loss:.4f}")
print(f"Loaded Model Test Accuracy: {loaded_acc:.4f}")

# -----------------------------------
# 11. TASK 7: PREDICTIONS
# -----------------------------------
predictions = loaded_model.predict(test_ds)
predicted_labels = np.argmax(predictions, axis=1)

true_labels = np.concatenate([y for x, y in test_ds], axis=0)

print("\nFirst 10 predicted labels:", predicted_labels[:10])
print("First 10 true labels:     ", true_labels[:10])

# -----------------------------------
# 12. CLASSIFICATION REPORT
# -----------------------------------
print("\nClassification Report:\n")
print(classification_report(
    true_labels,
    predicted_labels,
    target_names=train_ds.class_names
))

# -----------------------------------
# 13. SHOW SOME TEST IMAGES WITH PREDICTIONS
# -----------------------------------
test_images = []
test_actual = []

for batch_images, batch_labels in test_ds.take(1):
    test_images = batch_images.numpy()
    test_actual = batch_labels.numpy()

sample_preds = loaded_model.predict(batch_images)
sample_pred_labels = np.argmax(sample_preds, axis=1)

plt.figure(figsize=(14, 8))
num_show = min(10, len(test_images))

for i in range(num_show):
    plt.subplot(2, 5, i + 1)
    plt.imshow(test_images[i])
    pred_name = train_ds.class_names[sample_pred_labels[i]]
    true_name = train_ds.class_names[test_actual[i]]
    plt.title(f"P: {pred_name}\nT: {true_name}")
    plt.axis("off")

plt.tight_layout()
plt.show()